# StormEngine V8 — Frozen 2016 validation benchmark

This notebook compares V8 Stage 3A, V8 Stage 2, frozen V7-B, sparse reconstruction persistence, and dense ERA5 persistence on exactly the same 2016 windows. Event thresholds use 2010–2015 only. The evaluator never instantiates 2017.

In [ ]:
from pathlib import Path
import json, subprocess, sys
here = Path.cwd().resolve()
REPO = here if (here / 'pyproject.toml').exists() else here.parent
assert (REPO / 'pyproject.toml').exists(), REPO
DEVICE = 'cuda'
CONFIG = REPO / 'configs' / 'v8_2016_benchmark.yaml'
OUTPUT = REPO / 'artifacts' / 'v8_2016_benchmark'
print('Repository:', REPO)
print('Config:', CONFIG)

## 1. Checkpoint, cache, and tensor preflight

This verifies all five frozen checkpoint hashes and contracts, one shared 2016 batch, the 390-station order, and `test_years_read=[]`.

In [ ]:
preflight = [sys.executable, '-u', str(REPO / 'scripts' / 'evaluate_v8_2016_benchmark.py'), '--config', str(CONFIG), '--mode', 'preflight', '--device', DEVICE]
subprocess.run(preflight, cwd=REPO, check=True)

## 2. Complete frozen 2016 benchmark

This may take several model-evaluation passes but performs no training. Do not add `--max-batches` to the formal run.

In [ ]:
command = [sys.executable, '-u', str(REPO / 'scripts' / 'evaluate_v8_2016_benchmark.py'), '--config', str(CONFIG), '--mode', 'evaluate', '--device', DEVICE, '--publish-dir', 'results/v8_2016_validation_benchmark']
subprocess.run(command, cwd=REPO, check=True)

## 3. Inspect the pre-registered decision

In [ ]:
result = json.loads((OUTPUT / 'benchmark.json').read_text(encoding='utf-8'))
print(json.dumps({'scientific_status': result['scientific_status'], 'contract': result['contract'], 'event_thresholds': result['event_thresholds'], 'acceptance': result['acceptance']}, indent=2))